# Step 4: Structured Actions via Function Calling

**Topic:** Replacing Step 2's free-text JSON parsing with Groq function calling, validated against a Pydantic schema

**Resources used:**
- Groq tool use docs (console.groq.com/docs/tool-use/overview and /local-tool-calling), checked on 2026-09-02
- `qwen/qwen3.8-27b` supports local tool use (confirmed in Groq's supported-models table), same vision model as Step 2
- Phase 6 Step 4 pattern (`structured_extraction.ipynb`, browser-agent-from-scratch repo), Pydantic schema plus Groq tool calling, applied here to actions instead of data extraction

**Summary:** Defines a Pydantic action schema, forces the model to call a matching Groq tool instead of returning free text, validates and retries on malformed tool-call arguments, and wires the result into the Step 2 agent loop.


## Setup

Run this cell first. Requires `GROQ_API_KEY` in a `.env` file in the working directory, same as Steps 2 and 3.

In [1]:
!pip install pyautogui mss groq python-dotenv pydantic


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:

import os
import time
import json
import base64
from typing import Optional, Literal

import pyautogui
import mss
import mss.tools
from dotenv import load_dotenv
from groq import Groq
from groq import APIStatusError
from pydantic import BaseModel, Field, ValidationError

load_dotenv()
client = Groq(api_key=os.environ["GROQ_API_KEY"])

VISION_MODEL = "qwen/qwen3.8-27b"  # confirmed local tool use support, same model as Step 2


## Section 1: Pydantic action schema

`Literal` restricts `action` to exactly these five values, this is what makes the schema
enforceable rather than just documentation, both Pydantic and the JSON schema sent to
Groq will reject anything else. Fields that only apply to some actions stay `Optional`,
since a `scroll` action has no `text` and a `type` action has no `direction`.

In [3]:
class AgentAction(BaseModel):
    action: Literal["click", "type", "scroll", "wait", "done"] = Field(
        description="The single next action to perform"
    )
    x: Optional[int] = Field(default=None, description="X coordinate for click actions")
    y: Optional[int] = Field(default=None, description="Y coordinate for click actions")
    text: Optional[str] = Field(default=None, description="Text to type for type actions")
    direction: Optional[Literal["up", "down"]] = Field(
        default=None, description="Scroll direction for scroll actions"
    )
    amount: Optional[int] = Field(default=None, description="Scroll amount in clicks for scroll actions")
    reasoning: str = Field(description="Short reason for choosing this action")


Test the schema on its own with both a valid and an invalid payload.

In [4]:
valid_example = AgentAction(action="click", x=100, y=200, reasoning="clicking the submit button")
print(valid_example)

try:
    AgentAction(action="fly", reasoning="not a real action")
except ValidationError as e:
    print("Rejected as expected:")
    print(e)


action='click' x=100 y=200 text=None direction=None amount=None reasoning='clicking the submit button'
Rejected as expected:
1 validation error for AgentAction
action
  Input should be 'click', 'type', 'scroll', 'wait' or 'done' [type=literal_error, input_value='fly', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/literal_error


## Section 2: Function calling setup

`model_json_schema()` turns the Pydantic model straight into the JSON schema Groq's
`parameters` field expects, so the schema only has to be defined once instead of kept in
sync by hand in two places. `tool_choice` is set to force this exact function, matching
Groq's documented `{"type": "function", "function": {"name": ...}}` forced-call format,
this replaces Step 2's `response_format=json_object` with a stronger guarantee, the model
can't skip the tool and answer in free text.

In [5]:
ACTION_TOOL_SCHEMA = {
    "type": "function",
    "function": {
        "name": "submit_action",
        "description": "Submit the single next action to perform on the screen",
        "parameters": AgentAction.model_json_schema(),
    },
}

def ask_model_for_structured_action(base64_image, task_description, history=None, temperature=0.2):
    screen_width, screen_height = pyautogui.size()
    history = history or []

    prompt = (
        f"You control a computer by looking at screenshots. The screen is "
        f"{screen_width}x{screen_height} pixels, origin (0,0) is the top-left corner. "
        f"Task: {task_description}\n"
        f"Steps taken so far: {history}\n"
        "Call the submit_action tool with the single next action to take."
    )

    response = client.chat.completions.create(
        model=VISION_MODEL,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {
                        "type": "image_url",
                        "image_url": {"url": f"data:image/png;base64,{base64_image}"},
                    },
                ],
            }
        ],
        tools=[ACTION_TOOL_SCHEMA],
        tool_choice={"type": "function", "function": {"name": "submit_action"}},
        temperature=temperature,
        max_completion_tokens=512,
    )

    return response


## Section 3: Validate the tool call, retry on malformed arguments

Groq's docs specify that a malformed tool call surfaces as an HTTP 400 with a
`failed_generation` field, that is a request-level failure, not something present in a
normal response object, so it has to be caught as an exception around the API call
itself. A second, separate failure mode is a tool call that comes back as a 200 but with
arguments that don't satisfy the schema (e.g. `action: "click"` with no `x`/`y`), that is
caught as a Pydantic `ValidationError` after parsing. Both cases lower the temperature
and retry, per Groq's documented guidance that lower temperature reduces malformed
structured output on retry.

In [6]:
def get_validated_action(base64_image, task_description, history=None, max_retries=3):
    temperature = 0.2

    for attempt in range(1, max_retries + 1):
        try:
            response = ask_model_for_structured_action(
                base64_image, task_description, history, temperature=temperature
            )
        except APIStatusError as e:
            # this is Groq's documented malformed-tool-call case, a 400 with
            # failed_generation, not a normal response to parse
            print(f"Attempt {attempt}: API rejected the tool call ({e.status_code}), retrying")
            temperature = max(temperature - 0.1, 0.0)
            continue

        message = response.choices[0].message
        if not message.tool_calls:
            print(f"Attempt {attempt}: model did not call the tool, retrying")
            temperature = max(temperature - 0.1, 0.0)
            continue

        tool_call = message.tool_calls[0]

        try:
            raw_args = json.loads(tool_call.function.arguments)
        except json.JSONDecodeError:
            print(f"Attempt {attempt}: tool arguments were not valid JSON, retrying")
            temperature = max(temperature - 0.1, 0.0)
            continue

        try:
            return AgentAction.model_validate(raw_args)
        except ValidationError as e:
            print(f"Attempt {attempt}: tool arguments failed schema validation, retrying")
            print(e)
            temperature = max(temperature - 0.1, 0.0)
            continue

    raise RuntimeError(f"Failed to get a valid structured action after {max_retries} attempts")


Test the full validated call on its own before wiring it into the loop.

In [11]:
def capture_screenshot_base64():
    with mss.MSS() as sct:
        monitor = sct.monitors[1]
        shot = sct.grab(monitor)
        png_bytes = mss.tools.to_png(shot.rgb, shot.size)
    return base64.b64encode(png_bytes).decode("utf-8")


test_image = capture_screenshot_base64()
test_action = get_validated_action(test_image, "Just describe what you see and respond with action done.")
print(test_action)


action='done' x=None y=None text=None direction=None amount=None reasoning='Task only asks to describe the screen and finish; no further interaction is needed.'


## Section 4: Execute a structured action with PyAutoGUI

Same execution logic as Step 2, but reading from an `AgentAction` object with guaranteed
field types instead of a loosely parsed dict, so no `.get()` defensive lookups are needed
here, the schema already guarantees the shape.

In [13]:
def execute_structured_action(action: AgentAction):
    screen_width, screen_height = pyautogui.size()

    if action.action == "done":
        print("Model signaled done:", action.reasoning)
        return True

    if action.action == "click":
        if action.x is None or action.y is None:
            print("Skipping click, missing coordinates:", action)
            return False
        if not (0 <= action.x <= screen_width and 0 <= action.y <= screen_height):
            print("Skipping click, coordinates out of range:", action)
            return False
        pyautogui.moveTo(action.x, action.y, duration=0.3)
        pyautogui.click()

    elif action.action == "type":
        if action.text:
            pyautogui.write(action.text, interval=0.05)

    elif action.action == "scroll":
        clicks = action.amount or 1
        pyautogui.scroll(clicks if action.direction == "up" else -clicks)

    elif action.action == "wait":
        time.sleep(1)

    return False


## Section 5: Full loop wired to structured actions

Same shape as Step 2's `run_agent_loop`, `max_steps` is still the hard limit, but action
parsing is now `get_validated_action` instead of free-text `json.loads`, and history
entries are `model_dump()`'d dicts so they stay JSON-serializable for the next prompt.

In [14]:
def run_structured_agent_loop(task_description, max_steps=15, delay_between_steps=1.0):
    history = []

    for step in range(1, max_steps + 1):
        b64_image = capture_screenshot_base64()
        action = get_validated_action(b64_image, task_description, history)

        print(f"Step {step}: {action.action} -> {action.reasoning}")
        history.append(action.model_dump())

        is_done = execute_structured_action(action)
        if is_done:
            print(f"Loop finished after {step} step(s)")
            return history

        time.sleep(delay_between_steps)

    print(f"Hit max_steps ({max_steps}) without the model signaling done")
    return history


## Section 6: Free-text vs structured, a real reliability comparison

This reuses Step 2's free-text prompt-and-`json.loads` approach (no tool schema, no
retry) and runs it alongside this notebook's structured approach on the same
screenshot and prompt, `N_TRIALS` times each. Both call the real API, nothing here is
simulated, so the failure count reflects what actually happened on this run, not a
made-up number. Free-text failures here mean `json.loads` raised or a required key was
missing, since Step 2 had no schema to validate against.

In [15]:
def ask_model_free_text(base64_image, task_description):
    # same pattern as Step 2, response_format json_object but no schema enforcement
    # and no retry, this is deliberately the weaker baseline being compared against
    response = client.chat.completions.create(
        model=VISION_MODEL,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": (
                            f"Task: {task_description}\n"
                            'Respond with ONLY a JSON object: {"action": "...", "x": <int or null>, '
                            '"y": <int or null>, "text": <string or null>, "reasoning": "..."}'
                        ),
                    },
                    {
                        "type": "image_url",
                        "image_url": {"url": f"data:image/png;base64,{base64_image}"},
                    },
                ],
            }
        ],
        response_format={"type": "json_object"},
        temperature=0.2,
        max_completion_tokens=300,
    )
    return response.choices[0].message.content


N_TRIALS = 5
comparison_task = "Just describe what you see and respond with action done."
comparison_image = capture_screenshot_base64()

free_text_failures = 0
for i in range(N_TRIALS):
    raw = ask_model_free_text(comparison_image, comparison_task)
    try:
        data = json.loads(raw)
        assert "action" in data  # Step 2 never actually checked this, so check it now
    except (json.JSONDecodeError, AssertionError):
        free_text_failures += 1
        print(f"Free-text trial {i + 1}: parse failure")
    else:
        print(f"Free-text trial {i + 1}: parsed ok")

structured_failures = 0
for i in range(N_TRIALS):
    try:
        get_validated_action(comparison_image, comparison_task, max_retries=1)  # 1 try, no retry cushion
        print(f"Structured trial {i + 1}: valid")
    except RuntimeError:
        structured_failures += 1
        print(f"Structured trial {i + 1}: validation failure")

print(f"\nFree-text failures: {free_text_failures}/{N_TRIALS}")
print(f"Structured failures (no retry): {structured_failures}/{N_TRIALS}")


Free-text trial 1: parse failure
Free-text trial 2: parsed ok
Free-text trial 3: parse failure
Free-text trial 4: parse failure
Free-text trial 5: parsed ok
Structured trial 1: valid
Structured trial 2: valid
Structured trial 3: valid
Structured trial 4: valid
Structured trial 5: valid

Free-text failures: 3/5
Structured failures (no retry): 0/5
